# 022 — Training of EfficientNet models (two-phase fine-tuning)

Trains `efficientnet_unet`/`efficientnet_unet_ft` and `efficientnet_unet_nll`/`efficientnet_unet_nll_ft`.  
These architectures are built on a pretrained EfficientNetB0 encoder.

**Two-phase fine-tuning**: each model trains in two subprocess phases 
- phase 1 with the encoder frozen (`efficientnet_unet` / `efficientnet_unet_nll`)
- phase 2 warm-started from phase 1's weights with the encoder unfrozen (`efficientnet_unet_ft` / `efficientnet_unet_nll_ft`)

The two phases are saved as separate checkpoints so the frozen-encoder baseline is never overwritten.

Imports

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

import json
import subprocess

import matplotlib.pyplot as plt
import tensorflow as tf

from scripts.config import settings
from scripts.dataset import (
    build_dataset,
    load_image_pairs,
    mockup_aware_train_val_test_split,
)
from scripts.reproducibility import set_global_seed
from scripts.visualization import plot_training_curves

set_global_seed()

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs available: {gpus}")
print(f"TensorFlow version: {tf.__version__}")

## 1. Dataset split
**Split used project-wide:**  
- Real **artworks** are grouped and kept entirely within one fold, exactly as a plain grouped split would do, so no painting leaks across train/val/test.  
- The **mockup** groups (listed in `settings.MOCKUP_ARTWORK_IDS`) exist purely to be learned from. They are split at the individual-pair level, with only `settings.MOCKUP_TEST_RATIO` (default 5%) held out for test. 


In [ ]:
pairs = load_image_pairs(settings.IR_DIR, settings.RGB_DIR)
train_pairs, val_pairs, _ = mockup_aware_train_val_test_split(
    pairs,
    train_ratio=settings.TRAIN_RATIO,
    val_ratio=settings.VAL_RATIO,
    mockup_ids=settings.MOCKUP_ARTWORK_IDS,
    mockup_test_ratio=settings.MOCKUP_TEST_RATIO,
    seed=settings.SEED,
)

train_ds = build_dataset(
    train_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=True,
    shuffle=True,
    seed=settings.SEED,
    crop_size=settings.CROP_SIZE,
)
val_ds = build_dataset(
    val_pairs,
    batch_size=settings.BATCH_SIZE,
    augment=False,
    shuffle=False,
)

print(f"Train: {len(train_pairs)} patches ({len(train_ds)} batches)")
print(f"Val:   {len(val_pairs)} patches ({len(val_ds)} batches)")

## 2. Train the deterministic model

- Phase 1 trains `efficientnet_unet` with the encoder frozen and the decoder learns against fixed pretrained features.  
- Phase 2 unfreezes the encoder and continues training end-to-end.   

Each phase runs in its own subprocess.

**Note:** on first use downloads EfficientNetB0 ImageNet weights. Subsequent runs use the local Keras cache.

In [ ]:
EPOCHS = settings.EPOCHS  # -- lower for a quick smoke test
DET_MODEL_DIR = settings.MODELS_DIR / "deterministic"
DET_LOG_DIR = settings.LOGS_DIR / "deterministic"

det_histories: dict = {}

# Phase 1 (frozen encoder) -> "efficientnet_unet"; phase 2 (unfrozen,
# warm-started from phase 1) -> the separate "efficientnet_unet_ft"
# checkpoint (fixing.md #6).
DET_PHASES = [
    {
        "arch": "efficientnet_unet",
        "epochs": EPOCHS,
        "lr": settings.LEARNING_RATE,
        "kwargs": {},
        "init_from": None,
    },
    {
        "arch": "efficientnet_unet_ft",
        "epochs": settings.FINETUNE_EPOCHS,
        "lr": settings.FINETUNE_LEARNING_RATE,
        "kwargs": {"freeze_encoder": False},
        "init_from": DET_MODEL_DIR / "efficientnet_unet" / "best_model.keras",
    },
]

for phase in DET_PHASES:
    print(f"\n{'=' * 60}")
    print(f"  Architecture: {phase['arch']}")
    print(f"{'=' * 60}")

    cmd = [
        sys.executable,
        "-m",
        "scripts.train_single",
        "--arch",
        phase["arch"],
        "--epochs",
        str(phase["epochs"]),
        "--model-dir",
        str(DET_MODEL_DIR),
        "--log-dir",
        str(DET_LOG_DIR),
        "--kwargs",
        json.dumps(phase["kwargs"]),
        "--lr",
        str(phase["lr"]),
    ]
    if phase["init_from"] is not None:
        cmd += ["--init-from", str(phase["init_from"])]

    subprocess.run(cmd, cwd=project_root, check=True)

    history_path = DET_MODEL_DIR / phase["arch"] / "history.json"
    det_histories[phase["arch"]] = json.loads(history_path.read_text())

    best_val_loss = min(det_histories[phase["arch"]]["val_loss"])
    print(f"\nBest val_loss ({phase['arch']}): {best_val_loss:.4f}")

## 3. Train the NLL models

Same two-phase structure as for deterministic, for the heteroscedastic `(mu, log_b)`: 
- phase 1 trains `efficientnet_unet_nll` with the encoder frozen
- phase 2 (`efficientnet_unet_nll_ft`) unfreezes it

In [ ]:
NLL_MODEL_DIR = settings.MODELS_DIR / "nll"
NLL_LOG_DIR = settings.LOGS_DIR / "nll"

nll_histories: dict = {}

NLL_PHASES = [
    {
        "arch": "efficientnet_unet_nll",
        "epochs": EPOCHS,
        "lr": settings.LEARNING_RATE,
        "kwargs": {},
        "init_from": None,
    },
    {
        "arch": "efficientnet_unet_nll_ft",
        "epochs": settings.FINETUNE_EPOCHS,
        "lr": settings.FINETUNE_LEARNING_RATE,
        "kwargs": {"freeze_encoder": False},
        "init_from": NLL_MODEL_DIR / "efficientnet_unet_nll" / "best_model.keras",
    },
]

for phase in NLL_PHASES:
    print(f"\n{'=' * 60}")
    print(f"  Architecture: {phase['arch']}")
    print(f"{'=' * 60}")

    cmd = [
        sys.executable,
        "-m",
        "scripts.train_single",
        "--arch",
        phase["arch"],
        "--epochs",
        str(phase["epochs"]),
        "--model-dir",
        str(NLL_MODEL_DIR),
        "--log-dir",
        str(NLL_LOG_DIR),
        "--kwargs",
        json.dumps(phase["kwargs"]),
        "--lr",
        str(phase["lr"]),
        "--nll",
        "--loss-name",
        "laplace_nll",
    ]
    if phase["init_from"] is not None:
        cmd += ["--init-from", str(phase["init_from"])]

    subprocess.run(cmd, cwd=project_root, check=True)

    history_path = NLL_MODEL_DIR / phase["arch"] / "history.json"
    nll_histories[phase["arch"]] = json.loads(history_path.read_text())

    best_val_loss = min(nll_histories[phase["arch"]]["val_loss"])
    print(f"\nBest val_loss ({phase['arch']}): {best_val_loss:.4f}")

## 4. Training curves

In [ ]:
for arch, history in det_histories.items():
    plot_training_curves(history, title=f"Training history — {arch}")
    plt.show()

for arch, history in nll_histories.items():
    plot_training_curves(history, title=f"Training history — {arch}")
    plt.show()

## 5. Summary

In [ ]:
checks = [
    ("efficientnet_unet", DET_MODEL_DIR),
    ("efficientnet_unet_ft", DET_MODEL_DIR),
    ("efficientnet_unet_nll", NLL_MODEL_DIR),
    ("efficientnet_unet_nll_ft", NLL_MODEL_DIR),
]
for arch, model_dir in checks:
    ckpt = model_dir / arch / "best_model.keras"
    status = "found" if ckpt.exists() else "MISSING"
    print(f"{arch:<25} ({model_dir.name:<12}): {status}  ({ckpt})")